In [ ]:
import sys
print("The python version is: " + sys.version)

# AdvTG — end-to-end pipeline

Adversarial HTTP traffic generation vs. DL malicious-traffic detectors, run stage by stage:

1. **Dataset** → 2. **Detectors** (token + image) → 3. **LLM finetune** (QLoRA) → 4. **PPO** adversarial generation

**Where this runs:** a **Kaggle notebook** (GPU). The cells run on Kaggle's VM, not your laptop.

- Notebook ▸ Settings: **Accelerator = GPU T4 x2** (avoid P100: current torch/bitsandbytes builds don't support it), **Internet = On** (needed for git clone, pip, and HF model downloads).
- Runs on Kaggle's default Python. The stack is modern and unpinned, so you don't need a special runtime, a torch downgrade, or kernel restarts.
- The VM only has what's in the **git remote**. **Push your branch first**; the Setup cell clones it into `/kaggle/working/AdvTG` (and `git pull`s on re-runs). Local edits do **not** sync to the VM.
- Storage: the repo, `dataset/` and `model/` live under **`/kaggle/working`**, which is kept as notebook output when you *Save Version* (20 GB limit). HF model weights are cached outside it, so they don't count against that limit.
- Optional: add an **`HF_TOKEN`** under Add-ons ▸ Secrets. It gives faster, rate-limit-free HF downloads.
- Only GPU 0 is used (`CUDA_VISIBLE_DEVICES=0`). Everything fits on one T4, and this avoids multi-GPU surprises.

**Each stage installs its own deps** at the top of its first cell (there's no central install step), so you can run any stage on its own. `requirements.txt` still lists the full stack for venv/Docker use. Run **Setup + Config** once, then run the stages in order.

In [ ]:
import os, sys, subprocess

# Kaggle T4 x2: pin to one GPU (must be set before torch initialises CUDA).
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# HF token from Kaggle Secrets (Add-ons ▸ Secrets ▸ HF_TOKEN) — optional.
if "HF_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass   # no secret attached -> unauthenticated HF downloads

# Setup as a function. Light + idempotent (git pull + chdir + sys.path — NO torch import),
# so it's safe to call at the top of any stage via prepare(). Run this cell once to define it.
GIT_URL = "https://github.com/TejaswiMN/AdvTG.git"
BRANCH  = "LLM-Finetune"
WORK    = "/kaggle/working"            # persisted as notebook output on Save Version

def setup():
    """Clone/pull the repo, cd into it, make it importable. Sets global REPO."""
    global REPO
    if os.path.isfile(os.path.join(os.getcwd(), "gen_synthetic_data.py")):
        REPO = os.getcwd()                                   # already inside the repo
    else:
        # clone target on the hosted VM (Kaggle)
        REPO = os.path.join(WORK, "AdvTG")
        if os.path.isdir(os.path.join(REPO, ".git")):
            subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=False)
            subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=False)
            subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
        else:
            subprocess.run(["git", "clone", "-b", BRANCH, GIT_URL, REPO], check=True)
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)                             # make the DL package importable
    return REPO

setup()
print("repo:", REPO, "| has gen_synthetic_data.py:", os.path.isfile("gen_synthetic_data.py"))

## Config — the settings live in `config()`

To change a setting (dataset source, sizes, epochs…), edit the value **inside the `config()` function** below and re-run this cell. Everything downstream reads `dataset/train_data2.json`.

`setup()` and `config()` are light, torch-free helpers; `prepare()` runs both and is called at the top of every stage — so each stage is self-contained and you never have to scroll back up to re-run Setup/Config.

In [ ]:
import os

# Config as a function (light + idempotent, no torch import). prepare() = setup() + config(),
# called at the top of every stage so each stage is self-contained. To change a setting,
# edit the value inside config() and re-run this cell.
def config():
    """Define dataset/training settings + paths (needs REPO from setup())."""
    global DATASET_SOURCE, N_TRAIN, N_TEST, MALICIOUS_RATIO
    global HF_DATASET_REPO, HF_LLM_PCAPS, HF_LLM_MAX_REQUESTS
    global MAX_LENGTH, BATCH_SIZE, NUM_EPOCHS, TRAIN_BERT
    global DATA_DIR, MODEL_DIR, TRAIN_JSON, TEST_JSON

    # --- dataset settings -----------------------------------------------------
    DATASET_SOURCE  = "cicids2017"      # "synthetic" | "cicids2017"
    N_TRAIN         = 20000
    N_TEST          = 4000
    MALICIOUS_RATIO = 0.35
    # Stage 3 downloads exactly this one PCAP from Hugging Face.
    HF_DATASET_REPO = "bencorn/CICIDS2017"
    HF_LLM_PCAPS = [("Monday-WorkingHours.pcap", "Benign")]
    HF_LLM_MAX_REQUESTS = 10000
    # detector training
    MAX_LENGTH = 512
    BATCH_SIZE = 16
    NUM_EPOCHS = 2
    TRAIN_BERT = False
    # paths
    DATA_DIR   = os.path.join(REPO, "dataset")
    MODEL_DIR  = os.path.join(REPO, "model")
    TRAIN_JSON = os.path.join(DATA_DIR, "train_data2.json")
    TEST_JSON  = os.path.join(DATA_DIR, "test2.json")
    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)

def prepare():
    """setup() + config() — call once at the top of any stage."""
    setup(); config()

config()
print("dataset source:", DATASET_SOURCE, "| model dir:", MODEL_DIR)
print("LLM PCAPs:", HF_LLM_PCAPS, "| max requests:", HF_LLM_MAX_REQUESTS)

## Stage 1 — Dataset

Produces `dataset/train_data2.json` (+ `test2.json`) — the single file every downstream stage reads.

In [ ]:
import os
prepare()   # re-establish repo + config, so this stage runs standalone

if DATASET_SOURCE == "synthetic":
    !python gen_synthetic_data.py --n {N_TRAIN} --test-n {N_TEST} \
        --malicious-ratio {MALICIOUS_RATIO} --out "{TRAIN_JSON}" --test-out "{TEST_JSON}"

elif DATASET_SOURCE == "cicids2017":
    raise RuntimeError(
        "The CICIDS Stage 1 extractor was removed. Run Stage 3 directly; "
        "it downloads only HF_LLM_PCAPS (Monday-WorkingHours.pcap)."
    )

else:
    raise ValueError("DATASET_SOURCE must be 'synthetic' or 'cicids2017'")

In [ ]:
import json, collections
recs = json.load(open(TRAIN_JSON, encoding="utf-8"))
print(len(recs), "records", dict(collections.Counter(r["Label"] for r in recs)))
r = recs[0]
print("\n" + r["Request Line"])
for k, v in list(r["Request Headers"].items())[:4]:
    print(f"  {k}: {v}")
print("  ->", r["Label"], "| source:", r["Source"])

## Stage 2 — Detector training

Trains the token-level (TextCNN / CNN-LSTM / DNN) and image detectors, then writes the `model_configs` pickles the PPO stage attacks. Runs on CPU or GPU.

In [ ]:
# token-level detectors (TextCNN / CNN-LSTM / DNN), sharing the BERT tokenizer's vocab
prepare()   # re-establish repo + config (must run before the DL imports below)
# Stage 2 deps (torch/numpy come from the runtime):
!pip -q install "transformers>=4.46" "datasets>=2.20" "scikit-learn>=1.3"
import os, torch
import DL.training as _T
from transformers import TrainingArguments, AutoTokenizer
from DL.data_processing import load_data, prepare_dataset
from DL.models import TextCNNClassifier, CNNLSTMClassifier, DNNClassifier
from DL.training import train_custom_model, train_transformer_model

_T.MODEL_PATH = MODEL_DIR          # repo hardcodes ./models/; redirect saves under model/
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

TOKENIZER_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
data = load_data(TRAIN_JSON)
train_ds, val_ds, test_ds = prepare_dataset(data, tokenizer, MAX_LENGTH)

vocab_size, embed_size, num_classes = len(tokenizer.vocab), 128, 2
os.makedirs(os.path.join(MODEL_DIR, "custom_models"), exist_ok=True)
args = TrainingArguments(output_dir=os.path.join(MODEL_DIR, "custom_models"),
                         per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
                         learning_rate=2e-5, num_train_epochs=NUM_EPOCHS, report_to="none")

for name, model in {
        "textcnn":  TextCNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH),
        "cnn_lstm": CNNLSTMClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH),
        "dnn":      DNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)}.items():
    print("training", name)
    train_custom_model(model, name, train_ds, val_ds, args)

if TRAIN_BERT:
    bargs = TrainingArguments(output_dir=os.path.join(MODEL_DIR, "bert"),
                              evaluation_strategy="epoch", learning_rate=2e-5,
                              per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
                              num_train_epochs=NUM_EPOCHS, weight_decay=0.01, save_strategy="epoch",
                              load_best_model_at_end=True, report_to="none")
    train_transformer_model("bert", TOKENIZER_NAME, train_ds, val_ds, bargs)

In [ ]:
# image-based detectors: each request rendered as a 28x28 byte image (ord(c) % 128)
import numpy as np, torch, os
from torch.utils.data import DataLoader, TensorDataset
from DL.data_processing import json_to_string
from DL.image_models import ImageCNN, ImageMLP

IMG = (28, 28)
def to_image(it):
    text = it["Request Line"] + "\n" + json_to_string(it["Request Headers"]) + "\n\n" + it["Request Body"]
    v = [ord(c) % 128 for c in text][:IMG[0] * IMG[1]]
    v += [0] * (IMG[0] * IMG[1] - len(v))
    return np.array(v, dtype=np.float32).reshape(IMG)

X = torch.tensor(np.stack([to_image(r) for r in data]))
y = torch.tensor([1 if r["Label"] == "Malicious" else 0 for r in data], dtype=torch.long)
k = int(len(X) * 0.9)
loader = DataLoader(TensorDataset(X[:k], y[:k]), batch_size=64, shuffle=True)

for name, m in {"imagecnn": ImageCNN(), "imagemlp": ImageMLP()}.items():
    m.to(device); opt = torch.optim.Adam(m.parameters(), 1e-3); lf = torch.nn.CrossEntropyLoss()
    for _ in range(int(NUM_EPOCHS)):
        m.train()
        for xb, yb in loader:
            loss = lf(m(xb.to(device)), yb.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
    m.eval()
    with torch.no_grad():
        acc = (m(X[k:].to(device)).argmax(1).cpu() == y[k:]).float().mean().item()
    path = os.path.join(MODEL_DIR, "custom_models", name + ".bin")
    torch.save(m.state_dict(), path)
    print(f"{name}: val acc {acc:.3f} -> {path}")

In [ ]:
import pickle, os
from transformers import AutoTokenizer
from DL.models import TextCNNClassifier, CNNLSTMClassifier, DNNClassifier
from DL.image_models import ImageCNN, ImageMLP

cm = os.path.join(MODEL_DIR, "custom_models")

def cfg(name, cls):
    return {"type": "custom", "name": name, "path": os.path.join(cm, name + ".bin"), "class": cls}

text_configs = [
    cfg("textcnn",  TextCNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)),
    cfg("cnn_lstm", CNNLSTMClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)),
    cfg("dnn",      DNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)),
]
image_configs = [cfg("imagecnn", ImageCNN()), cfg("imagemlp", ImageMLP())]

pickle.dump(text_configs,  open(os.path.join(MODEL_DIR, "model_configs.pkl"), "wb"))
pickle.dump(image_configs, open(os.path.join(MODEL_DIR, "imgae_model_configs.pkl"), "wb"))

# PPO's Text reward re-tokenises responses with this — save it even when BERT is skipped.
AutoTokenizer.from_pretrained(TOKENIZER_NAME).save_pretrained(os.path.join(MODEL_DIR, "bert"))
print("wrote model_configs.pkl, imgae_model_configs.pkl, model/bert/ tokenizer")

## Stage 3 — LLM finetuning (QLoRA, GPU)

4-bit LoRA SFT of Llama-3-8b — plain **peft + bitsandbytes** (no unsloth) — to generate benign/malicious
traffic in the dataset's format. Fits a T4 (MAX_SEQ 1024, batch 1 + grad-accum). Uses TRL **`SFTConfig`**;
saves the adapter to `model/llama_lora`.

In [ ]:
!sudo DEBIAN_FRONTEND=noninteractive apt-get install -y tshark

In [ ]:
import os
import subprocess
import time
import torch

print("=" * 70)
print("STAGE 3 — QLoRA TRAINING")
print("=" * 70)

print("\n[1/9] Re-establishing repository/config...")
prepare()
print("      ✓ prepare() complete")

# -------------------------------------------------------------------
# INSTALL DEPENDENCIES
# -------------------------------------------------------------------

print("\n[2/9] Installing Stage 3 dependencies...")

!pip -q install "trl>=0.11" "peft>=0.12" "bitsandbytes>=0.43" "accelerate>=0.34" "huggingface_hub"

print("      ✓ Dependencies installed")

assert torch.cuda.is_available(), "Stage 3 needs a GPU runtime"

print("\nGPU:")
print("      Device:", torch.cuda.get_device_name(0))
print("      CUDA :", torch.version.cuda)
print("      BF16 :", torch.cuda.is_bf16_supported())

# -------------------------------------------------------------------
# IMPORTS
# -------------------------------------------------------------------

print("\n[3/9] Loading Python libraries...")

from huggingface_hub import hf_hub_download, login
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

print("      ✓ Libraries loaded")

# -------------------------------------------------------------------
# HUGGING FACE LOGIN
# -------------------------------------------------------------------

print("\n[4/9] Authenticating with Hugging Face...")

# Better: use Kaggle Secrets instead of putting the token directly here.
#
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
HF_TOKEN = ""
login(token=HF_TOKEN)

# If you already authenticated earlier in this notebook,
# you don't need to call login again.

print("      ✓ Hugging Face authentication ready")

# -------------------------------------------------------------------
# CONFIG
# -------------------------------------------------------------------

MODEL = "unsloth/llama-3-8b-bnb-4bit"
MAX_SEQ = 1024
_bf16 = torch.cuda.is_bf16_supported()

print("\nConfiguration:")
print("      Model       :", MODEL)
print("      Max sequence:", MAX_SEQ)
print("      BF16        :", _bf16)
print("      PCAPs       :", HF_LLM_PCAPS)
print("      Max requests:", HF_LLM_MAX_REQUESTS)

# -------------------------------------------------------------------
# PCAP DOWNLOAD
# -------------------------------------------------------------------

print("\n" + "=" * 70)
print("PCAP EXTRACTION")
print("=" * 70)


def download_pcap(filename):

    print(f"\n[PCAP] Looking for: {filename}")

    try:
        print("       Trying repository root...")
        path = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            repo_type="dataset",
            filename=filename
        )

        print("       ✓ Found at repository root")

    except Exception:
        print("       Not found at root.")
        print("       Trying pcaps/ directory...")

        path = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            repo_type="dataset",
            filename="pcaps/" + filename
        )

        print("       ✓ Found under pcaps/")

    size_gb = os.path.getsize(path) / (1024 ** 3)

    print(f"       Local path : {path}")
    print(f"       File size  : {size_gb:.2f} GB")

    return path


def pcap_requests(filename, label, limit):

    path = download_pcap(filename)

    print("\n" + "-" * 70)
    print(f"STARTING TSHARK")
    print(f"File  : {filename}")
    print(f"Label : {label}")
    print(f"Limit : {limit}")
    print("-" * 70)

    cmd = [
        "tshark",
        "-r", path,

        "-o", "tcp.desegment_tcp_streams:TRUE",
        "-o", "http.desegment_headers:TRUE",
        "-o", "http.desegment_body:TRUE",

        "-Y", "http.request",

        "-T", "fields",

        "-E", "separator=\t",
        "-E", "occurrence=f"
    ]

    for field in [
        "http.request.method",
        "http.request.uri",
        "http.request.version",
        "http.file_data"
    ]:
        cmd += ["-e", field]

    print("\n[TShark] Command prepared")
    print("[TShark] Starting process...")

    start_time = time.time()

    rows = []

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1 << 20
    )

    print(f"[TShark] PID: {proc.pid}")
    print("[TShark] RUNNING...")
    print("[TShark] Waiting for HTTP requests...\n")

    last_report = time.time()

    try:

        for raw in proc.stdout:

            values = raw.rstrip("\r\n").split("\t")
            values += [""] * (4 - len(values))

            method, uri, version, body = values[:4]

            if not method or not uri:
                continue

            # Convert hex HTTP body if necessary
            if body and ":" in body and " " not in body:

                try:
                    body = bytes.fromhex(
                        body.replace(":", "")
                    ).decode(
                        "utf-8",
                        "replace"
                    )

                except ValueError:
                    pass

            rows.append({
                "Request Line":
                    "%s %s %s" %
                    (
                        method,
                        uri,
                        version or "HTTP/1.1"
                    ),

                "Request Headers": {},

                "Request Body":
                    body,

                "Label":
                    label,

                "Source":
                    filename
            })

            # ---------------------------------------------------------
            # PERIODIC PROGRESS REPORT
            # ---------------------------------------------------------

            now = time.time()

            if now - last_report >= 5:

                elapsed = now - start_time

                print(
                    f"[TShark] Running... "
                    f"HTTP requests extracted: {len(rows):,} "
                    f"| elapsed: {elapsed / 60:.1f} min"
                )

                last_report = now

            # ---------------------------------------------------------
            # STOP WHEN ENOUGH REQUESTS ARE FOUND
            # ---------------------------------------------------------

            if len(rows) >= limit:

                print(
                    f"\n[TShark] ✓ Reached request limit "
                    f"({limit:,})"
                )

                print("[TShark] Terminating process...")

                proc.kill()

                break

    except KeyboardInterrupt:

        print("\n[TShark] Interrupted by user.")
        proc.kill()
        raise

    finally:

        proc.wait()

    elapsed = time.time() - start_time

    # Read stderr after process finishes
    stderr_output = proc.stderr.read().strip()

    print("\n" + "-" * 70)
    print("TSHARK FINISHED")
    print("-" * 70)

    print(f"Exit code        : {proc.returncode}")
    print(f"Requests extracted: {len(rows):,}")
    print(f"Runtime           : {elapsed / 60:.2f} minutes")

    if stderr_output:

        print("\n[TShark stderr]")
        print(stderr_output[:2000])

    if proc.returncode not in (0, -9):

        raise RuntimeError(
            f"tshark failed for {filename}: "
            f"{stderr_output}"
        )

    return rows


# -------------------------------------------------------------------
# EXTRACT DATA
# -------------------------------------------------------------------

print("\n[5/9] Extracting HTTP requests from PCAPs...")

data = []

for i, (pcap_name, pcap_label) in enumerate(
    HF_LLM_PCAPS,
    start=1
):

    print("\n")
    print("#" * 70)
    print(
        f"PCAP {i}/{len(HF_LLM_PCAPS)}"
    )
    print("#" * 70)

    extracted = pcap_requests(
        pcap_name,
        pcap_label,
        HF_LLM_MAX_REQUESTS
    )

    data.extend(extracted)

    print(
        f"\n[PCAP {i}] Total dataset rows so far: "
        f"{len(data):,}"
    )


if not data:

    raise RuntimeError(
        "No HTTP requests found. "
        "Choose a PCAP containing plaintext HTTP traffic."
    )

print("\n✓ PCAP extraction complete")
print(f"✓ Total HTTP requests: {len(data):,}")

# -------------------------------------------------------------------
# LOAD MODEL
# -------------------------------------------------------------------

print("\n" + "=" * 70)
print("MODEL LOADING")
print("=" * 70)

print("\n[6/9] Loading tokenizer...")

tok = AutoTokenizer.from_pretrained(MODEL)

tok.pad_token = tok.eos_token

print("      ✓ Tokenizer loaded")
print("      Vocabulary size:", len(tok))

print("\n[6/9] Loading 4-bit Llama model...")
print("      This may take several minutes.")

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    dtype=torch.float16
)

print("      ✓ Base model loaded")

print("\n[6/9] Preparing model for QLoRA...")

model = prepare_model_for_kbit_training(model)

print("      ✓ k-bit preparation complete")

# -------------------------------------------------------------------
# LORA
# -------------------------------------------------------------------

print("\n[7/9] Applying LoRA configuration...")

model = get_peft_model(
    model,
    LoraConfig(
        r=16,
        lora_alpha=16,
        lora_dropout=0.0,
        bias="none",
        task_type="CAUSAL_LM",

        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj"
        ]
    )
)

model.print_trainable_parameters()

print("      ✓ LoRA applied")

# -------------------------------------------------------------------
# DATASET
# -------------------------------------------------------------------

print("\n[8/9] Preparing training dataset...")


def json_to_string(d, indent=0):

    out = []
    pad = " " * indent

    if isinstance(d, dict):

        for k, v in d.items():

            if isinstance(v, (dict, list)):

                out.append(
                    f"{pad}{k}:"
                )

                out.append(
                    json_to_string(
                        v,
                        indent + 1
                    )
                )

            else:

                out.append(
                    f"{pad}{k}: {v}"
                )

    elif isinstance(d, list):

        for it in d:

            out.append(
                json_to_string(
                    it,
                    indent
                )
            )

    else:

        out.append(
            f"{pad}{d}"
        )

    return "\n".join(out)


alpaca = (
    "Below is an instruction that describes a task, paired "
    "with an input that provides further context. Write a "
    "response that appropriately completes the request.\n\n"
    "### Instruction:\n{}\n\n"
    "### Input:\n{}\n\n"
    "### Response:\n{}"
)

EOS = tok.eos_token


def body(it):

    return (
        it["Request Line"]
        + "\n"
        + json_to_string(
            it["Request Headers"]
        )
        + "\n\n"
        + it["Request Body"]
    )


texts = []

for i, it in enumerate(data):

    instruction = (
        "Follow these tips to generate malicious http traffic"
        if it["Label"] == "Malicious"
        else
        "Follow these tips to generate benign http traffic"
    )

    text = alpaca.format(
        instruction,
        it["Request Line"],
        body(it)
    ) + EOS

    texts.append(text)

    if (i + 1) % 500 == 0:

        print(
            f"      Formatted {i + 1:,}/{len(data):,} examples"
        )


ds = Dataset.from_dict(
    {"text": texts}
).shuffle(seed=42)

print("\n      ✓ Dataset created")
print("      Examples:", len(ds))

# Show one example
print("\n      Example training record:")
print("-" * 50)
print(ds[0]["text"][:1000])
print("-" * 50)

# -------------------------------------------------------------------
# TRAINER
# -------------------------------------------------------------------

print("\n[9/9] Creating SFT trainer...")

output_dir = os.path.join(
    MODEL_DIR,
    "llama_outputs"
)

trainer = SFTTrainer(
    model=model,
    processing_class=tok,
    train_dataset=ds,

    args=SFTConfig(

        dataset_text_field="text",
        max_length=MAX_SEQ,
        packing=False,

        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,

        warmup_steps=5,
        max_steps=60,

        learning_rate=2e-4,

        fp16=not _bf16,
        bf16=_bf16,

        gradient_checkpointing=True,

        gradient_checkpointing_kwargs={
            "use_reentrant": False
        },

        logging_steps=5,

        optim="paged_adamw_8bit",

        weight_decay=0.01,

        lr_scheduler_type="linear",

        seed=3407,

        output_dir=output_dir,

        report_to="none"
    )
)

print("      ✓ Trainer created")

# -------------------------------------------------------------------
# TRAIN
# -------------------------------------------------------------------

print("\n" + "=" * 70)
print("STARTING QLORA TRAINING")
print("=" * 70)

print("Steps              :", 60)
print("Batch size         :", 1)
print("Gradient accumulation:", 16)
print("Effective batch    :", 16)
print("Max sequence       :", MAX_SEQ)
print("Learning rate      :", 2e-4)

training_start = time.time()

trainer.train()

training_time = time.time() - training_start

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    f"Training time: {training_time / 60:.2f} minutes"
)

# -------------------------------------------------------------------
# SAVE
# -------------------------------------------------------------------

print("\nSaving LoRA adapter...")

lora_dir = os.path.join(
    MODEL_DIR,
    "llama_lora"
)

model.save_pretrained(lora_dir)
tok.save_pretrained(lora_dir)

print("\n" + "=" * 70)
print("✓ STAGE 3 COMPLETE")
print("=" * 70)

print("LoRA saved to:")
print(lora_dir)

In [ ]:
# -------------------------------------------------------------------
# HUGGING FACE LOGIN
# -------------------------------------------------------------------

print("\nAuthenticating with Hugging Face...")

# Better: use Kaggle Secrets instead of putting the token directly here.
#
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
HF_TOKEN = ""
login(token=HF_TOKEN)

# If you already authenticated earlier in this notebook,
# you don't need to call login again.

print("      ✓ Hugging Face authentication ready")
HF_REPO = "Aviral20/AdvTG-Llama3-8B-LoRA"

model.push_to_hub(
    HF_REPO,
    token=HF_TOKEN
)

tok.push_to_hub(
    HF_REPO,
    token=HF_TOKEN
)

print("Uploaded successfully!")
print(f"https://huggingface.co/{HF_REPO}")

## Stage 4 — PPO adversarial generation (GPU)

Hand-rolled PPO in `RL-Adv/ppo_core.py` (**no trl-PPO** — immune to trl API churn) tunes a generator
to flip the frozen detectors' predictions. Reward = mean detector probability of the *opposite*
label. `policy="pythia"` is the cheap demo generator; **Stage 4b** runs the Stage-3 Llama (paper-faithful).

In [ ]:
!pip uninstall -y bitsandbytes
!pip install -q --no-cache-dir "bitsandbytes>=0.46.1"

In [ ]:
import sys
import torch
import bitsandbytes as bnb

print("Python:", sys.version)
print("bitsandbytes:", bnb.__version__)
print("bnb location:", bnb.__file__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

print("=" * 60)
print("Downloading Stage-3 LoRA from Hugging Face")
print("=" * 60)

# Get HF token from Kaggle Secrets
from huggingface_hub import login

HF_TOKEN=""
login(token=HF_TOKEN)

HF_REPO = "Aviral20/AdvTG-Llama3-8B-LoRA"

# This is the directory ppo_core.py already expects
LOCAL_LORA_DIR = os.path.join(
    REPO,
    "model",
    "llama_lora"
)

print("HF repo :", HF_REPO)
print("Local   :", LOCAL_LORA_DIR)

os.makedirs(LOCAL_LORA_DIR, exist_ok=True)

snapshot_download(
    repo_id=HF_REPO,
    repo_type="model",
    local_dir=LOCAL_LORA_DIR,
    token=HF_TOKEN
)

print("\n✓ LoRA downloaded successfully")
print("\nFiles:")

for f in os.listdir(LOCAL_LORA_DIR):
    print("  ", f)

In [ ]:
import os, sys, importlib

print("=" * 70)
print("STAGE 4 — PPO Adervserial Generation")
print("=" * 70)

prepare()

# ------------------------------------------------------------
# Repository / paths
# ------------------------------------------------------------

RL_DIR = os.path.join(REPO, "RL-Adv")

sys.path.insert(0, RL_DIR)
os.chdir(RL_DIR)

print("REPO :", REPO)
print("RL   :", RL_DIR)
print("CWD  :", os.getcwd())

# ------------------------------------------------------------
# Reload PPO code
# ------------------------------------------------------------

import ppo_core
importlib.reload(ppo_core)

print("✓ ppo_core loaded")

# ------------------------------------------------------------
# Verify LoRA exists
# ------------------------------------------------------------

LORA_DIR = os.path.join(REPO, "model", "llama_lora")

print("\nLoRA directory:")
print(LORA_DIR)

if not os.path.exists(LORA_DIR):
    raise FileNotFoundError(
        f"LoRA directory not found: {LORA_DIR}"
    )

print("✓ LoRA directory exists")

print("\nLoRA files:")
for f in os.listdir(LORA_DIR):
    print("   ", f)

# ------------------------------------------------------------
# Stage 4
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STARTING LLAMA LoRA PPO")
print("=" * 70)

asr = ppo_core.train_ppo(
    policy="llama",          # IMPORTANT: llama, not pythia
    feature_type="Text",

    steps=40,
    sample_size=2000,

    # Llama 8B is much heavier than Pythia 160M
    batch_size=1,

    # Explicitly tell ppo_core where the downloaded adapter is
    lora_dir="../model/llama_lora"
)

# ------------------------------------------------------------
# Return to repository root
# ------------------------------------------------------------

os.chdir(REPO)

print("\n" + "=" * 70)
print("STAGE 4 COMPLETE")
print("=" * 70)

print("Stage 4 (llama) ASR:", asr)

In [ ]:
import os, sys, importlib
prepare()   # re-establish repo + config, so this stage runs standalone
# deps already installed by Stage 2 (transformers, datasets); torch comes from the runtime.
RL_DIR = os.path.join(REPO, "RL-Adv")
sys.path.insert(0, RL_DIR)
os.chdir(RL_DIR)                       # so ppo_core's ../model and ../dataset paths resolve
import ppo_core; importlib.reload(ppo_core)

# policy="pythia" -> cheap demo generator (EleutherAI/pythia-160m). FEATURE_TYPE: "Text" | "Image".
# Tune the knobs (steps / sample_size / batch_size) as needed; batch_size must be 1..4.
asr = ppo_core.train_ppo(policy="pythia", feature_type="Text",
                         steps=40, sample_size=2000, batch_size=4)
os.chdir(REPO)
print("Stage 4 (pythia) ASR:", asr)

In [ ]:
import os
for p in [os.path.join(REPO, "dataset", "train_data2.json"),
          os.path.join(REPO, "model", "model_configs.pkl"),
          os.path.join(REPO, "model", "llama_lora", "adapter_config.json")]:
    print(os.path.exists(p), p)

In [ ]:
import glob, json, os

files = sorted(glob.glob(os.path.join(REPO, "dataset", "PPO_data", "**", "*.json"), recursive=True))
if files:
    print("latest:", files[-1])
    print(json.dumps(json.load(open(files[-1]))[:2], indent=2)[:1500])
else:
    print("no PPO_data yet — run Stage 4 first")

## Stage 4b (faithful): PPO-tune the Stage-3 Llama instead of pythia-160m

In [ ]:
import torch
print(f"GPU used: {torch.cuda.memory_allocated()/1e9:.1f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
import os
lora = os.path.join(REPO, "model", "llama_lora")
ckpt = os.path.join(REPO, "model", "ppo_llama_ckpt")
print("Stage 3 LoRA present:", os.path.isfile(os.path.join(lora, "adapter_config.json")))
print("  contents:", os.listdir(lora) if os.path.isdir(lora) else "(missing)")
print("PPO checkpoint present:", os.path.isdir(ckpt) and os.path.exists(os.path.join(ckpt, "progress.json")))

In [ ]:
# ── Stage 4b (faithful): PPO-tune the Stage-3 Llama (policy="llama") ──
# Needs model/llama_lora from Stage 3 on disk. Checkpoints to model/ppo_llama_ckpt (auto-resume
# on re-run). Heavier than pythia — tiny defaults "prove it runs"; raise steps/sample_size and use
# a bigger GPU for paper-grade. Same hand-rolled loop as Stage 4, just policy="llama".
import os, sys, importlib
prepare()
# deps already installed by Stage 3 (peft + bitsandbytes), whose model/llama_lora this consumes.
RL_DIR = os.path.join(REPO, "RL-Adv")
sys.path.insert(0, RL_DIR)
os.chdir(RL_DIR)
import ppo_core; importlib.reload(ppo_core)
asr = ppo_core.train_ppo(policy="llama", feature_type="Text",
                         steps=15, sample_size=256, batch_size=2)
os.chdir(REPO)
print("Stage 4b (llama) ASR:", asr)